# ANTARES / LSST Alert Distribution Comparison

Compares alert distributions from the [ANTARES](https://antares.noirlab.edu) broker across **two MJD time ranges**:
- **Range 1** - Last night: a snapshot of what LSST observed most recently
- **Range 2** - Full LSST history: the cumulative population of all LSST-observed objects

All heavy lifting (queries, lightcurve fetching, caching, plotting, validation) lives in the `src/` package. This notebook is a thin storyboard that wires those helpers together.

Visualisations produced (in later cells):
1. Sky distribution - Aitoff projection (equatorial)
2. Sky density - RA/Dec 2-D histogram
3. Magnitude histograms - 15 < mag < 25, log-scale y-axis, per ZTF filter (g / r / i)

> **MJD rule**: if MJDmin > MJDmax for either range, that range is skipped and nothing is plotted for it.

In [ ]:
# Colab bootstrap: make sure the helper package exists in the runtime.
# If this notebook is opened directly from GitHub, Colab may not place the
# whole repository on disk. This clones/updates it automatically.
from pathlib import Path

REPO_URL = 'https://github.com/darim1151/ANTARES_Analysis.git'
COLAB_REPO = Path('/content/ANTARES_Analysis')

if 'google.colab' in str(get_ipython()):
    if not (COLAB_REPO / 'src' / 'chunked_query.py').exists():
        %cd /content
        !rm -rf ANTARES_Analysis
        !git clone {REPO_URL}
    else:
        %cd {COLAB_REPO}
        !git pull --ff-only
    %cd {COLAB_REPO}
else:
    print('Local runtime detected; using current working directory.')

## 1. Install dependencies (Colab; harmless to re-run locally)

In [ ]:
!pip install --quiet antares-client elasticsearch-dsl astropy matplotlib pandas numpy pyarrow

## 2. Imports and path setup

We add the project root to `sys.path` so `from src import ...` works regardless of where Jupyter was launched from.

In [ ]:
import sys
from pathlib import Path

# Project root = parent of this notebook's folder. Inserting it on
# sys.path lets us `from src import ...` whether the notebook is
# launched from the repo root or from inside notebooks/.
if 'COLAB_REPO' in globals() and COLAB_REPO.exists():
    PROJECT_ROOT = COLAB_REPO
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Dark theme matches the colour choices made in src/figures.py.
plt.style.use('dark_background')
%matplotlib inline

from src import config, query, chunked_query, lightcurves, cache, summary, figures, validation

print('Imports complete.')
print(f'  matplotlib {matplotlib.__version__}')
print(f'  pandas     {pd.__version__}')
print(f'  numpy      {np.__version__}')

## 3. Configuration

All knobs live in `src/config.py`. Edit there to change MJD windows, sample size, tag filter, or the random seed.

In [ ]:
config.print_config_summary()

## 4. Validate MJD ranges

If `MJDmin > MJDmax` for a range it is marked invalid and silently skipped by every downstream step.

In [ ]:
print('MJD Validation')
print('=' * 55)
RANGE1_VALID = config.validate_mjd_range(config.LABEL1, config.MJD1_MIN, config.MJD1_MAX)
RANGE2_VALID = config.validate_mjd_range(config.LABEL2, config.MJD2_MIN, config.MJD2_MAX)

if not RANGE1_VALID and not RANGE2_VALID:
    raise ValueError('Both MJD ranges are invalid. Edit MJD1/MJD2 in src/config.py.')

## 5. Cache check

If a previous run produced parquet files for the SAME parameters, load from disk and skip ANTARES entirely. Chunked runs use a separate cache directory and also maintain a cumulative loci store for nightly updates.

In [ ]:
USE_CACHE = True   # load from cache if available
SAVE_CACHE = True  # write cache after a live fetch

# Keep sampled and chunked cache snapshots separate so old sampled
# runs never masquerade as complete chunked ingestion results.
CACHE_PROFILE = 'chunked' if config.USE_CHUNKED_INGEST else 'sampled'
CACHE_DIR = str(PROJECT_ROOT / 'data' / f'cache_{CACHE_PROFILE}')
CHUNK_CACHE_DIR = str(PROJECT_ROOT / 'data' / 'chunk_cache')
CUMULATIVE_LOCI_PATH = str(PROJECT_ROOT / 'data' / 'antares_cumulative_loci.parquet')
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(CHUNK_CACHE_DIR).mkdir(parents=True, exist_ok=True)

_paths = cache.cache_paths(
    CACHE_DIR,
    config.MJD1_MIN, config.MJD1_MAX,
    config.MJD2_MIN, config.MJD2_MAX,
    config.N_SAMPLES,
)
_CACHE_LOADED, df1, df2, df1_alerts, df2_alerts = cache.try_load_cache(
    _paths, config.LABEL1, config.LABEL2, use_cache=USE_CACHE,
)

## 6. Run ANTARES queries (skipped if loaded from cache)

When `USE_CHUNKED_INGEST` is on, Range 1 is fetched through adaptive MJD chunks and appended into the cumulative loci store. Range 2 is then read from that store; if no history exists yet, the notebook falls back to the old sampled query unless `CHUNKED_BACKFILL_HISTORY` is enabled.

In [ ]:
if _CACHE_LOADED:
    print('Queries: skipped (data loaded from cache).')
else:
    if config.USE_CHUNKED_INGEST:
        print('Loading locus-level data with adaptive chunked ingestion')
        print('=' * 55)
        chunk_reports = {}

        if RANGE1_VALID:
            df1, chunk_reports['range1'] = chunked_query.query_range_adaptive(
                config.LABEL1, config.MJD1_MIN, config.MJD1_MAX,
                tag=config.QUERY_TAG,
                initial_chunk_days=config.CHUNK_INITIAL_DAYS,
                min_chunk_seconds=config.CHUNK_MIN_SECONDS,
                max_results_per_chunk=config.CHUNK_MAX_RESULTS,
                split_threshold=config.CHUNK_SPLIT_THRESHOLD,
                chunk_cache_dir=CHUNK_CACHE_DIR,
                use_chunk_cache=USE_CACHE,
            )
        else:
            df1 = pd.DataFrame()

        # Load historical comparison data before appending Range 1, so
        # tonight's loci do not leak into the "history" comparison sample.

        if RANGE2_VALID:
            if config.CHUNKED_BACKFILL_HISTORY:
                df2, chunk_reports['range2'] = chunked_query.query_range_adaptive(
                    config.LABEL2, config.MJD2_MIN, config.MJD2_MAX,
                    tag=config.QUERY_TAG,
                    initial_chunk_days=config.CHUNK_INITIAL_DAYS,
                    min_chunk_seconds=config.CHUNK_MIN_SECONDS,
                    max_results_per_chunk=config.CHUNK_MAX_RESULTS,
                    split_threshold=config.CHUNK_SPLIT_THRESHOLD,
                    chunk_cache_dir=CHUNK_CACHE_DIR,
                    use_chunk_cache=USE_CACHE,
                )
                _, store_stats = chunked_query.update_cumulative_loci(CUMULATIVE_LOCI_PATH, df2)
                print(f"  Cumulative loci store after history backfill: {store_stats['stored_rows']:,} rows")
            else:
                df2 = chunked_query.load_cumulative_range(
                    CUMULATIVE_LOCI_PATH, config.MJD2_MIN, config.MJD2_MAX,
                )
                if df2.empty:
                    print('  No cumulative history available for Range 2; falling back to sampled query.')
                    df2 = query.query_range(
                        config.LABEL2, config.MJD2_MIN, config.MJD2_MAX,
                        n_samples=config.N_SAMPLES, tag=config.QUERY_TAG, seed=config.RANDOM_SEED,
                    )
        else:
            df2 = pd.DataFrame()

        _, store_stats = chunked_query.update_cumulative_loci(CUMULATIVE_LOCI_PATH, df1)
        print(f"  Cumulative loci store after nightly update: {store_stats['stored_rows']:,} rows  ({CUMULATIVE_LOCI_PATH})")
    else:
        print('Loading locus-level data (sampled; no lightcurves - fast)')
        print('=' * 55)

        # Build per-range kwargs; pass None to skip an invalid range entirely.
        r1_args = dict(
            label=config.LABEL1, mjd_min=config.MJD1_MIN, mjd_max=config.MJD1_MAX,
            n_samples=config.N_SAMPLES, tag=config.QUERY_TAG, seed=config.RANDOM_SEED,
        ) if RANGE1_VALID else None
        r2_args = dict(
            label=config.LABEL2, mjd_min=config.MJD2_MIN, mjd_max=config.MJD2_MAX,
            n_samples=config.N_SAMPLES, tag=config.QUERY_TAG, seed=config.RANDOM_SEED,
        ) if RANGE2_VALID else None

        df1, df2 = query.query_both_ranges_parallel(r1_args, r2_args)

    print(f'  {config.LABEL1:20s}: {len(df1):>5d} loci')
    print(f'  {config.LABEL2:20s}: {len(df2):>5d} loci')

## 7. Load lightcurves (per-alert photometry)

Each lightcurve is one HTTP request, so we use a thread pool. Set `LOAD_LIGHTCURVES = False` to skip and fall back to locus-level magnitudes.

In [ ]:
LOAD_LIGHTCURVES = True
N_LC_SAMPLES = 50000  # lightcurves to fetch per range

if _CACHE_LOADED:
    print('Lightcurves: skipped (data loaded from cache).')
elif LOAD_LIGHTCURVES:
    print('Loading lightcurves (per-alert data)')
    print('=' * 55)
    df1_alerts = lightcurves.load_lightcurves(df1, N_LC_SAMPLES, config.LABEL1)
    df2_alerts = lightcurves.load_lightcurves(df2, N_LC_SAMPLES, config.LABEL2)
else:
    print('Lightcurve loading skipped (LOAD_LIGHTCURVES = False).')
    print('Magnitude histograms will use locus-level brightest_alert_magnitude.')

## 8. Save cache (only after a fresh fetch)

In [ ]:
if not _CACHE_LOADED and SAVE_CACHE:
    cache.save_cache(_paths, df1, df2, df1_alerts, df2_alerts)
elif _CACHE_LOADED:
    print('Cache was used - no save needed.')

## 9. Summary statistics

In [ ]:
summary.print_all_summaries(
    config.LABEL1, df1, df1_alerts,
    config.LABEL2, df2, df2_alerts,
)

## 10. Plot 1 - Sky distribution (Aitoff)

In [ ]:
figures.plot_sky_aitoff(
    df1, df2, RANGE1_VALID, RANGE2_VALID,
    config.LABEL1, config.LABEL2,
    config.MJD1_MIN, config.MJD2_MAX,
);

## 11. Plot 2 - Sky density (2-D RA/Dec histogram)

In [ ]:
figures.plot_sky_density(
    df1, df2, RANGE1_VALID, RANGE2_VALID,
    config.LABEL1, config.LABEL2,
);

## 12. Plot 3 - Magnitude histograms

In [ ]:
figures.plot_magnitude_histograms(
    df1, df2, df1_alerts, df2_alerts,
    RANGE1_VALID, RANGE2_VALID,
    config.LABEL1, config.LABEL2,
);

## 13. Validation suite

Eight numbered tests covering data presence, MJD compliance, coordinate sanity, LSST-era origin, magnitude window coverage, locus uniqueness, cross-range overlap (a critical randomisation check), and the MJDmin/MJDmax guard.

In [ ]:
validation.run_validation_suite(
    df1, df2, df1_alerts, df2_alerts,
    RANGE1_VALID, RANGE2_VALID,
    config.LABEL1, config.LABEL2,
    config.MJD1_MIN, config.MJD1_MAX,
    config.MJD2_MIN, config.MJD2_MAX,
);